<a href="https://colab.research.google.com/github/chiyanglin-AStar/2025_physics_note/blob/main/Cupy_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## 對話紀錄

**使用者 (Initial request):** 做 在Cupy 的tutorial ,也要有安裝步驟 其中的範例 要能表現 cupy 的 特性 , 要加上程式說明 要把對話紀錄記錄在筆記本裡

**AI:** 好的，這是一個關於 CuPy 的教學，包含了安裝步驟、展示 CuPy 特性的範例，以及程式碼說明。我會將這段對話記錄在筆記本中。

**執行結果:**
*   `!nvidia-smi` 顯示 `command not found`。
*   CuPy 已成功安裝 (`cupy-cuda12x-14.0.1`)。
*   CuPy 範例因未偵測到 GPU 而未能執行 (輸出: "沒有可用的 GPU，無法執行 CuPy 範例。")。

**AI:** 從您執行的結果來看，Colab 環境目前沒有偵測到可用的 GPU (`!nvidia-smi` 顯示 `command not found`)。因此，CuPy 範例無法在 GPU 上執行，並顯示了「沒有可用的 GPU，無法執行 CuPy 範例。」的訊息。這導致 CuPy 在 GPU 上的效能提升未能展現。

**要解決這個問題，您可以嘗試以下步驟：**
1.  **更改執行時期類型 (Runtime Type):** 在 Colab 菜單欄中，點擊 `執行階段 (Runtime)` -> `變更執行時期類型 (Change runtime type)`。
2.  在 `硬體加速器 (Hardware accelerator)` 選項中，選擇 `GPU`，然後點擊 `儲存 (Save)`。
3.  重新執行所有儲存格 (Runtime -> Run all)，特別是安裝 CuPy 和執行範例的儲存格。這樣應該就能讓 CuPy 範例在 GPU 上運行，並展示其加速效果了。


## CuPy 教學

您好！這是一個 CuPy 的教學，它將引導您完成安裝並透過範例了解 CuPy 的主要特性。

### 1. 安裝 CuPy

CuPy 是一個在 GPU 上實現 NumPy 陣列的函式庫。要使用 CuPy，您需要一個 NVIDIA GPU 並安裝 CUDA Toolkit。在 Colab 環境中，通常已經具備這些條件，您可以直接透過 `pip` 安裝 CuPy。

In [1]:
# 檢查 Colab 是否有可用的 GPU
!nvidia-smi

# 安裝 CuPy
# 如果您的環境已經安裝過 CUDA toolkit，可以直接安裝 CuPy
# 請注意，CuPy 的版本需要與您的 CUDA 版本相容
# 在 Colab 中通常安裝最新版本即可
!pip install cupy-cuda12x # 或者根據您的 CUDA 版本選擇 cupy-cuda11x 等

Fri Apr  3 23:25:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 2. CuPy 特性展示：與 NumPy 的效能比較

CuPy 的主要優勢在於其能夠利用 GPU 的並行運算能力來加速數值計算，特別是對於大型陣列操作。以下範例將展示 CuPy 相對於 NumPy 在矩陣乘法上的速度提升。

In [2]:
import numpy as np
import cupy as cp
import time

# 設定矩陣大小
n = 1000

# --- NumPy 範例 ---
print(f"進行 {n}x{n} 矩陣的 NumPy 運算...")
start_time = time.time()
A_np = np.random.rand(n, n).astype(np.float32)
B_np = np.random.rand(n, n).astype(np.float32)
C_np = np.dot(A_np, B_np)
end_time = time.time()
print(f"NumPy 運算時間: {end_time - start_time:.4f} 秒\n")

# --- CuPy 範例 ---
# 檢查是否有可用的 GPU
if cp.cuda.is_available():
    print(f"進行 {n}x{n} 矩陣的 CuPy (GPU) 運算...")
    start_time = time.time()
    A_cp = cp.asarray(A_np) # 將 NumPy 陣列複製到 GPU
    B_cp = cp.asarray(B_np)
    C_cp = cp.dot(A_cp, B_cp)
    cp.cuda.Stream.null.synchronize() # 等待 GPU 運算完成
    end_time = time.time()
    print(f"CuPy 運算時間: {end_time - start_time:.4f} 秒\n")
    # 將結果複製回 CPU 如果需要
    # C_host = cp.asnumpy(C_cp)
else:
    print("沒有可用的 GPU，無法執行 CuPy 範例。")


進行 1000x1000 矩陣的 NumPy 運算...
NumPy 運算時間: 0.0536 秒

進行 1000x1000 矩陣的 CuPy (GPU) 運算...
CuPy 運算時間: 0.3520 秒



### 3. CuPy 特性展示：元素級運算與聚合 (Element-wise Operations and Reductions)

除了矩陣乘法，CuPy 在處理大型陣列的元素級運算（如加、減、乘、平方等）以及聚合操作（如求和、平均值等）時，也能提供顯著的加速。以下範例將對一個大型陣列進行平方後求和的運算，並比較 NumPy 和 CuPy 的執行時間。

In [3]:
import numpy as np
import cupy as cp
import time

# 設定陣列大小
array_size = 50_000_000 # 5千萬個元素

print(f"進行 {array_size} 個元素的陣列運算 (平方後求和)...")

# --- NumPy 範例 ---
print("NumPy (CPU) 運算...")
start_time = time.time()
A_np_elem = np.random.rand(array_size).astype(np.float32)
B_np_elem = np.square(A_np_elem) # 元素級平方
C_np_elem = np.sum(B_np_elem)    # 聚合求和
end_time = time.time()
print(f"NumPy 運算時間: {end_time - start_time:.4f} 秒\n")

# --- CuPy 範例 ---
if cp.cuda.is_available():
    print("CuPy (GPU) 運算...")
    start_time = time.time()
    A_cp_elem = cp.asarray(A_np_elem) # 將 NumPy 陣列複製到 GPU
    B_cp_elem = cp.square(A_cp_elem)  # 元素級平方 (在 GPU 上執行)
    C_cp_elem = cp.sum(B_cp_elem)     # 聚合求和 (在 GPU 上執行)
    cp.cuda.Stream.null.synchronize() # 等待 GPU 運算完成
    end_time = time.time()
    print(f"CuPy 運算時間: {end_time - start_time:.4f} 秒\n")
    # 如果需要將結果複製回 CPU
    # C_host_elem = cp.asnumpy(C_cp_elem)
else:
    print("沒有可用的 GPU，無法執行 CuPy 元素級運算範例。")


進行 50000000 個元素的陣列運算 (平方後求和)...
NumPy (CPU) 運算...
NumPy 運算時間: 0.6068 秒

CuPy (GPU) 運算...
CuPy 運算時間: 0.6006 秒



### 程式碼說明 (元素級運算與聚合)

- `array_size = 50_000_000`: 設定了一個包含 5,000 萬個元素的大型一維陣列，這樣可以更明顯地看出 GPU 加速的優勢。
- `np.square(A_np_elem)` / `cp.square(A_cp_elem)`: 執行陣列中每個元素的平方運算。這是一個典型的元素級操作，在 CuPy 中它會在 GPU 上高效執行。
- `np.sum(B_np_elem)` / `cp.sum(B_cp_elem)`: 計算陣列所有元素的總和。這是一個聚合操作，CuPy 同樣能在 GPU 上快速完成。

這個範例進一步展示了 CuPy 不僅在複雜的線性代數運算（如矩陣乘法）中表現出色，在基本的元素級操作和聚合操作中，面對大數據量時也能提供卓越的性能。再次強調，要看到 CuPy 的性能優勢，請確保您的 Colab 環境已啟用 GPU。

如果您尚未啟用 GPU，請務必按照前面的說明，在 Colab 菜單欄中選擇 `執行階段 (Runtime)` -> `變更執行時期類型 (Change runtime type)`，然後在 `硬體加速器 (Hardware accelerator)` 選項中選擇 `GPU`，並重新執行所有儲存格。

### 程式碼說明

- `!nvidia-smi`: 執行這個命令可以檢查當前 Colab 環境中是否有可用的 NVIDIA GPU，並顯示其狀態。這是確認 CuPy 能否運行的重要一步。
- `!pip install cupy-cuda12x`: 這是安裝 CuPy 的命令。`cupy-cuda12x` 代表針對 CUDA 12.x 版本編譯的 CuPy。在 Colab 中，通常會預裝最新的 CUDA，所以這個版本通常是合適的。如果遇到問題，可能需要根據 `nvidia-smi` 輸出的 CUDA 版本調整。
- `import numpy as np` 和 `import cupy as cp`: 分別導入 NumPy 和 CuPy 函式庫，並給予慣用的別名。
- `n = 1000`: 設定了矩陣的維度，這裡創建的是 1000x1000 的矩陣。對於較小的矩陣，GPU 的優勢可能不明顯，甚至會因為數據傳輸的開銷而變慢，但對於大型矩陣，GPU 的加速效果會非常顯著。
- `np.random.rand(n, n).astype(np.float32)`: 使用 NumPy 創建一個 `n x n` 的隨機浮點數矩陣。`.astype(np.float32)` 確保數據類型為單精度浮點數，這在 GPU 運算中通常效率更高。
- `np.dot(A_np, B_np)`: 使用 NumPy 執行矩陣乘法。
- `cp.asarray(A_np)`: 這是 CuPy 的關鍵操作之一，它將一個 NumPy 陣列從 CPU 記憶體複製到 GPU 記憶體，並返回一個 CuPy 陣列 (ndarray)。所有的 GPU 運算都必須在 CuPy 陣列上進行。
- `cp.dot(A_cp, B_cp)`: 使用 CuPy 執行矩陣乘法。這個操作會在 GPU 上完成。
- `cp.cuda.Stream.null.synchronize()`: 這個命令非常重要。由於 GPU 運算是非同步的，`cp.dot` 可能會立即返回，但實際的運算仍在 GPU 後台進行。`synchronize()` 會阻塞 CPU，直到所有之前的 GPU 運算都完成，這樣我們才能準確測量 GPU 運算的時間。
- `cp.asnumpy(C_cp)`: 如果您需要將 GPU 上運算結果的 CuPy 陣列複製回 CPU 進行後續處理或顯示，可以使用此函數。

從上述範例的執行結果中，您應該會看到 CuPy (GPU) 的矩陣乘法運算時間顯著少於 NumPy (CPU) 的運算時間，這正是 CuPy 利用 GPU 加速的典型表現。
